In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/mle-practice-test/sample_submission.csv
/kaggle/input/competitions/mle-practice-test/train.csv
/kaggle/input/competitions/mle-practice-test/test.csv


In [2]:

import glob, os, pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import *

# Load data
tr = pd.read_csv("/kaggle/input/competitions/mle-practice-test/train.csv")
te = pd.read_csv("/kaggle/input/competitions/mle-practice-test/test.csv")

# Prepare data
X = tr.drop(columns=["id","label"]).copy()
y = tr["label"]
Xt = te[X.columns].copy()
ids = te["id"]

cat = X.select_dtypes("object").columns.tolist()
for c in cat:
    X[c] = X[c].fillna("Missing").astype(str)
    Xt[c] = Xt[c].fillna("Missing").astype(str)

# Split data
a,b,ya,yb = train_test_split(X,y,test_size=.2,random_state=42)

# Train model
m = CatBoostClassifier(verbose=0)
m.fit(a,ya,cat_features=cat,eval_set=(b,yb),early_stopping_rounds=100)

# Evaluate
prob = m.predict_proba(b)[:,1]
pred = (prob>=.5).astype(int)

print("Accuracy:",accuracy_score(yb,pred))
print("Precision:",precision_score(yb,pred,zero_division=0))
print("Recall:",recall_score(yb,pred,zero_division=0))
print("F1 Score:",f1_score(yb,pred))
print("ROC-AUC:",roc_auc_score(yb,prob))
print("Confusion Matrix:\n",confusion_matrix(yb,pred))

# Feature importance
print("\nFeature Importance:\n",
      pd.Series(m.feature_importances_,index=X.columns).sort_values(ascending=False))

# Final training
m.fit(X,y,cat_features=cat)

# Create submission
sub = pd.DataFrame({
    "id":ids,
    "label":m.predict(Xt)
})

sub.to_csv("/kaggle/working/submission.csv",index=False)
print("\nSubmission saved successfully!")
print(sub.head())


Accuracy: 0.9953125
Precision: 0.9956521739130435
Recall: 0.9913419913419913
F1 Score: 0.9934924078091106
ROC-AUC: 0.999894156373374
Confusion Matrix:
 [[408   1]
 [  2 229]]

Feature Importance:
 vitamin_b12_percent_rda    38.515687
disease_diagnosis          37.353106
iron_percent_rda            4.645814
folate_percent_rda          3.981664
symptoms_list               2.768617
serum_vitamin_b12_pg_ml     2.344926
symptoms_count              1.920317
vitamin_a_percent_rda       1.189682
has_numbness_tingling       0.877214
vitamin_d_percent_rda       0.743592
has_fatigue                 0.732842
serum_vitamin_d_ng_ml       0.718545
latitude_region             0.585807
sun_exposure                0.507218
diet_type                   0.505688
income_level                0.466034
vitamin_c_percent_rda       0.433667
serum_folate_ng_ml          0.274046
hemoglobin_g_dl             0.252528
exercise_level              0.227854
bmi                         0.190100
smoking_status            

In [ ]:
iuii